# Práctica: Embeddings y Vector Database — Azure AI Search

**Índice:** `rag-v1`  
**Servicio:** `vector-searcher`  
**Modelo de embeddings:** `text-embedding-ada-002` (1536 dimensiones)

Este notebook ejecuta los 4 tipos de búsqueda requeridos y un extra con Scoring Profile:
1. Vector Search
2. Hybrid Search (vector + keyword)
3. Semantic Search
4. Semantic Hybrid Search
5. ⭐ Extra — Scoring Profile

## 0. Instalación de dependencias

In [1]:
%pip install azure-search-documents openai python-dotenv --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Configuración — carga de variables desde `.env`

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # Carga el archivo .env

# Azure AI Search
SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
SEARCH_API_KEY  = os.getenv("AZURE_SEARCH_API_KEY")
INDEX_NAME      = os.getenv("AZURE_SEARCH_INDEX_NAME")

# Azure OpenAI
OPENAI_ENDPOINT    = os.getenv("AZURE_OPENAI_ENDPOINT")
OPENAI_API_KEY     = os.getenv("AZURE_OPENAI_API_KEY")
EMBEDDING_DEPLOY   = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

print(f"Endpoint Search : {SEARCH_ENDPOINT}")
print(f"Índice          : {INDEX_NAME}")
print(f"Endpoint OpenAI : {OPENAI_ENDPOINT}")
print(f"Deployment      : {EMBEDDING_DEPLOY}")

Endpoint Search : https://vector-searcher.search.windows.net
Índice          : rag-v1
Endpoint OpenAI : https://ai-sergiorincon0872ai045504941434.openai.azure.com
Deployment      : text-embedding-ada-002


## 2. Inicialización de clientes

In [3]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
from openai import AzureOpenAI

# Cliente de búsqueda
search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=AzureKeyCredential(SEARCH_API_KEY)
)

# Cliente de embeddings
openai_client = AzureOpenAI(
    azure_endpoint=OPENAI_ENDPOINT,
    api_key=OPENAI_API_KEY,
    api_version=OPENAI_API_VERSION
)

print("Clientes inicializados correctamente.")

Clientes inicializados correctamente.


## 3. Función auxiliar: generar embedding de una query

In [15]:
def get_embedding(text: str) -> list[float]:
    """Genera el vector embedding de un texto usando Azure OpenAI."""
    response = openai_client.embeddings.create(
        input=text,
        model=EMBEDDING_DEPLOY
    )
    return response.data[0].embedding


def print_results(results, search_type: str):
    """Imprime los resultados de búsqueda de forma legible."""
    print(f"\n{'='*60}")
    print(f"  {search_type}")
    print(f"{'='*60}")

    # Materializamos el iterador antes de recorrerlo
    items = list(results)

    if not items:
        print("  Sin resultados.")
        return

    for count, r in enumerate(items, start=1):
        score    = r.get('@search.score')
        reranker = r.get('@search.reranker_score')
        title    = r.get('title') or '(sin título)'
        chunk    = (r.get('chunk') or '')[:300]

        score_str    = f"{score:.4f}"    if score    is not None else "N/A"
        reranker_str = f"{reranker:.4f}" if reranker is not None else None

        print(f"\n[{count}] Score: {score_str}", end="")
        if reranker_str:
            print(f" | Reranker: {reranker_str}", end="")
        print(f"\n    Título : {title}")
        print(f"    Chunk  : {chunk}...")

    print(f"\nTotal resultados: {len(items)}")

## 4. Query de prueba

Modifica esta variable para probar con distintas preguntas.

In [5]:
QUERY = "¿Cuáles son los principales conceptos de machine learning?"

# Generamos el embedding una sola vez y lo reutilizamos en todas las búsquedas
query_vector = get_embedding(QUERY)
print(f"Query    : {QUERY}")
print(f"Dimensiones del vector: {len(query_vector)}")

Query    : ¿Cuáles son los principales conceptos de machine learning?
Dimensiones del vector: 1536


---
## 5. Vector Search

Búsqueda **puramente vectorial**: compara el embedding de la query con los vectores `text_vector` del índice usando similitud coseno (HNSW). No usa keywords ni texto.

**Cuándo usarla:** Cuando quieres encontrar contenido semánticamente similar aunque no comparta palabras exactas con la query.

In [7]:
vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=5,
    fields="text_vector"
)

results = search_client.search(
    search_text=None,           # Sin texto — solo vector
    vector_queries=[vector_query],
    select=["chunk_id", "title", "chunk"],
    top=5
)

print_results(results, "1. VECTOR SEARCH")


  1. VECTOR SEARCH

[1] Score: 0.8339
    Título : CV.pdf
    Chunk  : Ingeniería de Datos e IA Generativa, principalmente en entornos cloud.</p><p><br></p><p>Español - Inglés</p>
        
            
                FOUNDEVER
                
                    
                        
                            Madrid
                            es
              ...

[2] Score: 0.8273
    Título : CV.pdf
    Chunk  : SOBRE MÍ

Profesional híbrido con base en desarrollo de software y especialización en Arquitectura de Datos e
Inteligencia Artificial. Orientado a la resolución práctica de problemas mediante Big Data, Machine
Learning, Ingeniería de Datos e IA Generativa, principalmente en entornos cloud.

Español ...

[3] Score: 0.8262
    Título : CV.pdf
    Chunk  : Máster en Inteligencia Artificial & Big Data Tajamar. (En curso) (Sep. 2025 – Jun. 2026) 
                    
                        
                            IA y Big Data
                        
             

---
## 6. Hybrid Search (vector + keyword)

Combina búsqueda vectorial con búsqueda de texto completo (BM25). Azure AI Search usa **Reciprocal Rank Fusion (RRF)** para fusionar los scores de ambas.

**Cuándo usarla:** En la mayoría de casos de producción — combina lo mejor de la recuperación semántica y la keyword-based.

In [8]:
vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=5,
    fields="text_vector"
)

results = search_client.search(
    search_text=QUERY,          # BM25 sobre campos searchable
    vector_queries=[vector_query],
    select=["chunk_id", "title", "chunk"],
    top=5
)

print_results(results, "2. HYBRID SEARCH (vector + keyword)")


  2. HYBRID SEARCH (vector + keyword)

[1] Score: 0.0328
    Título : CV.pdf
    Chunk  : SOBRE MÍ

Profesional híbrido con base en desarrollo de software y especialización en Arquitectura de Datos e
Inteligencia Artificial. Orientado a la resolución práctica de problemas mediante Big Data, Machine
Learning, Ingeniería de Datos e IA Generativa, principalmente en entornos cloud.

Español ...

[2] Score: 0.0325
    Título : CV.pdf
    Chunk  : (vacío y nitrógeno líquido).


	Sergio
        Rincón De La Cruz
	Máster en Inteligencia Artificial & Big Data Tajamar. (En curso) (Sep. 2025 – Jun. 2026)
	Grado Superior en Desarrollo de Aplicaciones Web (DAW)
	Grado Medio en Sistemas Microinformáticos y Redes (SMR)
	Intérprete de Asistencia en Car...

[3] Score: 0.0323
    Título : CV.pdf
    Chunk  : Ingeniería de Datos e IA Generativa, principalmente en entornos cloud.</p><p><br></p><p>Español - Inglés</p>
        
            
                FOUNDEVER
                
                    
  

---
## 7. Semantic Search

Aplica el **reranker semántico** de Microsoft sobre los resultados de BM25. El modelo de lenguaje re-puntúa los candidatos según la comprensión del lenguaje natural, generando un `reranker_score`.

**Configuración semántica usada:** `rag-v1-semantic-configuration`  
- Campo título: `title`  
- Campo contenido: `chunk`

**Cuándo usarla:** Cuando la precisión del ranking es crítica y los documentos tienen texto rico.

In [9]:
results = search_client.search(
    search_text=QUERY,
    query_type="semantic",
    semantic_configuration_name="rag-v1-semantic-configuration",
    select=["chunk_id", "title", "chunk"],
    top=5
)

print_results(results, "3. SEMANTIC SEARCH")


  3. SEMANTIC SEARCH

[1] Score: 4.2916 | Reranker: 1.8944
    Título : CV.pdf
    Chunk  : (vacío y nitrógeno líquido).


	Sergio
        Rincón De La Cruz
	Máster en Inteligencia Artificial & Big Data Tajamar. (En curso) (Sep. 2025 – Jun. 2026)
	Grado Superior en Desarrollo de Aplicaciones Web (DAW)
	Grado Medio en Sistemas Microinformáticos y Redes (SMR)
	Intérprete de Asistencia en Car...

[2] Score: 3.6611 | Reranker: 1.8210
    Título : CV.pdf
    Chunk  : SOBRE MÍ

Profesional híbrido con base en desarrollo de software y especialización en Arquitectura de Datos e
Inteligencia Artificial. Orientado a la resolución práctica de problemas mediante Big Data, Machine
Learning, Ingeniería de Datos e IA Generativa, principalmente en entornos cloud.

Español ...

[3] Score: 2.0049 | Reranker: 1.6994
    Título : CV.pdf
    Chunk  : Ingeniería de Datos e IA Generativa, principalmente en entornos cloud.</p><p><br></p><p>Español - Inglés</p>
        
            
                FOUNDEVER


---
## 8. Semantic Hybrid Search

Combina los tres mecanismos: **BM25 + vector (RRF)** seguido de **reranking semántico**. Es el modo más potente disponible en Azure AI Search.

Pipeline:
1. BM25 recupera candidatos por keyword
2. HNSW recupera candidatos por vector
3. RRF fusiona ambas listas
4. El reranker semántico re-puntúa el top-N final

**Cuándo usarla:** Casos RAG donde la calidad del contexto recuperado impacta directamente en la respuesta del LLM.

In [10]:
vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=5,
    fields="text_vector"
)

results = search_client.search(
    search_text=QUERY,
    vector_queries=[vector_query],
    query_type="semantic",
    semantic_configuration_name="rag-v1-semantic-configuration",
    select=["chunk_id", "title", "chunk"],
    top=5
)

print_results(results, "4. SEMANTIC HYBRID SEARCH")


  4. SEMANTIC HYBRID SEARCH

[1] Score: 0.0325 | Reranker: 1.8944
    Título : CV.pdf
    Chunk  : (vacío y nitrógeno líquido).


	Sergio
        Rincón De La Cruz
	Máster en Inteligencia Artificial & Big Data Tajamar. (En curso) (Sep. 2025 – Jun. 2026)
	Grado Superior en Desarrollo de Aplicaciones Web (DAW)
	Grado Medio en Sistemas Microinformáticos y Redes (SMR)
	Intérprete de Asistencia en Car...

[2] Score: 0.0328 | Reranker: 1.8210
    Título : CV.pdf
    Chunk  : SOBRE MÍ

Profesional híbrido con base en desarrollo de software y especialización en Arquitectura de Datos e
Inteligencia Artificial. Orientado a la resolución práctica de problemas mediante Big Data, Machine
Learning, Ingeniería de Datos e IA Generativa, principalmente en entornos cloud.

Español ...

[3] Score: 0.0323 | Reranker: 1.6994
    Título : CV.pdf
    Chunk  : Ingeniería de Datos e IA Generativa, principalmente en entornos cloud.</p><p><br></p><p>Español - Inglés</p>
        
            
                FOU

---
## ⭐ 9. EXTRA — Scoring Profile

### ¿Qué es un Scoring Profile?

Un **Scoring Profile** modifica el score BM25 base de cada documento aplicando **boosts** sobre campos o funciones de valor numérico/fecha. Permite que el negocio influya en el ranking sin cambiar la query.

### Configuración

Vamos a añadir un scoring profile al índice `rag-v1` que haga boost del campo `title` (peso x3) y del campo `chunk` (peso x1.5).

Esto prioriza documentos donde la query aparece en el título sobre los que solo la contienen en el cuerpo.

### Paso 1 — Añadir el Scoring Profile al índice via REST API

In [11]:
import requests
import json

# Definición del scoring profile
scoring_profile = {
    "name": "title-boost-profile",
    "text": {
        "weights": {
            "title": 3.0,    # Triplica el score si la query aparece en el título
            "chunk": 1.5     # 1.5x si aparece en el contenido del chunk
        }
    }
}

# Actualizamos el índice añadiendo el scoring profile
# Usamos PATCH (merge) para no sobreescribir el resto de la config
url = f"{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}?api-version=2024-07-01"

headers = {
    "Content-Type": "application/json",
    "api-key": SEARCH_API_KEY
}

# Leemos el índice actual
get_resp = requests.get(url, headers=headers)
index_def = get_resp.json()

# Eliminamos el etag para poder hacer PUT
index_def.pop("@odata.etag", None)

# Añadimos el scoring profile si no existe ya
existing_profiles = [p["name"] for p in index_def.get("scoringProfiles", [])]
if "title-boost-profile" not in existing_profiles:
    index_def.setdefault("scoringProfiles", []).append(scoring_profile)
    put_resp = requests.put(url, headers=headers, json=index_def)
    print(f"PUT status: {put_resp.status_code}")
    if put_resp.status_code in (200, 204):
        print("✅ Scoring profile 'title-boost-profile' añadido correctamente.")
    else:
        print(f"❌ Error: {put_resp.text}")
else:
    print("ℹ️ El scoring profile ya existe en el índice.")

PUT status: 204
✅ Scoring profile 'title-boost-profile' añadido correctamente.


### Paso 2 — Búsqueda con el Scoring Profile activo

Comparamos la misma query **sin** y **con** el scoring profile para ver el efecto en el ranking.

In [12]:
# Sin scoring profile (BM25 puro)
results_base = search_client.search(
    search_text=QUERY,
    select=["chunk_id", "title", "chunk"],
    top=5
)
print_results(results_base, "5a. HYBRID SIN Scoring Profile (BM25 base)")


  5a. HYBRID SIN Scoring Profile (BM25 base)

[1] Score: 4.2916
    Título : CV.pdf
    Chunk  : (vacío y nitrógeno líquido).


	Sergio
        Rincón De La Cruz
	Máster en Inteligencia Artificial & Big Data Tajamar. (En curso) (Sep. 2025 – Jun. 2026)
	Grado Superior en Desarrollo de Aplicaciones Web (DAW)
	Grado Medio en Sistemas Microinformáticos y Redes (SMR)
	Intérprete de Asistencia en Car...

[2] Score: 3.6611
    Título : CV.pdf
    Chunk  : SOBRE MÍ

Profesional híbrido con base en desarrollo de software y especialización en Arquitectura de Datos e
Inteligencia Artificial. Orientado a la resolución práctica de problemas mediante Big Data, Machine
Learning, Ingeniería de Datos e IA Generativa, principalmente en entornos cloud.

Español ...

[3] Score: 3.4034
    Título : CV.pdf
    Chunk  : y coordinación en tiempo real. Interpretación simultánea en
llamadas a tres partes.

PAWIFY 

Población: Madrid |  País: España 

[ 01/03/2025 – 27/06/2025 ]  Prácticas Software Developer 



In [13]:
# Con scoring profile activo
results_boosted = search_client.search(
    search_text=QUERY,
    scoring_profile="title-boost-profile",
    select=["chunk_id", "title", "chunk"],
    top=5
)
print_results(results_boosted, "5b. HYBRID CON Scoring Profile 'title-boost-profile'")


  5b. HYBRID CON Scoring Profile 'title-boost-profile'

[1] Score: 6.4374
    Título : CV.pdf
    Chunk  : (vacío y nitrógeno líquido).


	Sergio
        Rincón De La Cruz
	Máster en Inteligencia Artificial & Big Data Tajamar. (En curso) (Sep. 2025 – Jun. 2026)
	Grado Superior en Desarrollo de Aplicaciones Web (DAW)
	Grado Medio en Sistemas Microinformáticos y Redes (SMR)
	Intérprete de Asistencia en Car...

[2] Score: 5.4916
    Título : CV.pdf
    Chunk  : SOBRE MÍ

Profesional híbrido con base en desarrollo de software y especialización en Arquitectura de Datos e
Inteligencia Artificial. Orientado a la resolución práctica de problemas mediante Big Data, Machine
Learning, Ingeniería de Datos e IA Generativa, principalmente en entornos cloud.

Español ...

[3] Score: 5.1051
    Título : CV.pdf
    Chunk  : y coordinación en tiempo real. Interpretación simultánea en
llamadas a tres partes.

PAWIFY 

Población: Madrid |  País: España 

[ 01/03/2025 – 27/06/2025 ]  Prácticas Software De

### Análisis del efecto del Scoring Profile

| Aspecto | Sin Scoring Profile | Con Scoring Profile |
|---|---|---|
| Score base | BM25 estándar | BM25 × boost por campo |
| Campo `title` | Peso = 1 | Peso = 3.0 |
| Campo `chunk` | Peso = 1 | Peso = 1.5 |
| Efecto | Ranking por frecuencia/rareza del término | Documentos con query en el título suben en el ranking |
| Caso de uso | Búsqueda general | Cuando el título es indicativo del tema principal del documento |

> **Nota:** Los scoring profiles solo afectan a la búsqueda de texto (BM25). En búsquedas puramente vectoriales no tienen efecto. En hybrid search afectan a la rama BM25 antes de la fusión RRF.

---
## 10. Resumen comparativo de los 4 tipos de búsqueda

| Tipo | `search_text` | `vector_queries` | `query_type` | Score principal |
|---|---|---|---|---|
| Vector Search | `None` | ✅ | — | Similitud coseno |
| Hybrid Search | ✅ | ✅ | — | RRF (BM25 + vector) |
| Semantic Search | ✅ | ❌ | `semantic` | BM25 → reranker score |
| Semantic Hybrid | ✅ | ✅ | `semantic` | RRF → reranker score |